# VRP 04 — Conditional short volatility: straddle and iron fly

**Discussion:** `discussions/Volatility_Risk_Premium/009_claude_execution_and_conditional_short.md` · **Code:** `vrp_conditional_short.py`

Sell volatility only when implied variance is high relative to a causal forecast:
$$w=\ln\frac{\sigma_{\text{straddle}}^2\,T}{\hat v_t\sum_{j\in(t,T]}s^2(\text{slot}_j)}$$
- $\hat v_t$ is the mean of 1h/4h/24h EMAs of de-clocked squared index returns.
- $s^2$ is the development-only time-of-day clock.

Structures, held to the delivery price:
- short ATM straddle;
- **short iron fly**: wings at the listed strikes nearest $F e^{\pm 1.5\sigma\sqrt T}$.

Prices are executable (sell bid / buy ask) and mid; fees per leg.

**Modes**
- **Development** (default): sessions < 2026-11-07; tercile thresholds computed in-sample. Descriptive.
- **Freeze** (`VRP04_FREEZE=1`, once, after 2026-11-06): writes `04/04_frozen_spec.json`; never overwritten.
- **Confirm** (`VRP04_CONFIRM=1`): sessions ≥ 2026-11-07 with the frozen spec. Binding at ≥ 20 sessions. Decision: top-tercile short iron fly at executable prices, return on risk, session-bootstrap CI > 0 → `CONDITIONAL_PREMIUM_CONFIRMED`.

Also: `VRP04_OFFLINE=1`, `VRP04_N_BOOT`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import vrp_long_straddle as ls
VRP = PROJECT_ROOT / 'notebooks' / 'Volatility_Risk_Premium'
from data_engineering.modeling import vrp_conditional_short as cs
FREEZE = os.getenv('VRP04_FREEZE', '').strip() == '1'
CONFIRM = os.getenv('VRP04_CONFIRM', '').strip() == '1'
OFFLINE = os.getenv('VRP04_OFFLINE', '').strip() == '1'
N_BOOT = int(os.getenv('VRP04_N_BOOT', str(cs.N_BOOT)))
if FREEZE and CONFIRM:
    raise RuntimeError('Choose one mode')
OUT = VRP / '04'
OUT.mkdir(parents=True, exist_ok=True)
SPEC = OUT / '04_frozen_spec.json'
MODE = 'confirm' if CONFIRM else ('freeze' if FREEZE else 'development')
if CONFIRM and not SPEC.exists():
    raise RuntimeError('Confirmation requires 04_frozen_spec.json (run the freeze first)')
print('VRP 04', cs.VERSION, cs.selftest(), '| mode:', MODE)
print(json.dumps(cs.registration(), indent=2))
T0 = time.time()

VRP 04 0.1.0 {'version': '0.1.0', 'ok': True} | mode: development
{
  "version": "0.1.0",
  "confirmation_start": "2026-11-07",
  "half_lives_bars": [
    12,
    48,
    288
  ],
  "min_history_bars": 576,
  "z_wing": 1.5,
  "n_boot": 4000,
  "seed": 9004,
  "min_confirm_sessions": 20,
  "decision": "confirmation only: top-tercile short iron fly at executable prices, session-bootstrap CI of mean RoR"
}


In [2]:
def load_chain(cols_wanted, where_extra, hourly):
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        have = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
        need = [c for c in ls.CHAIN_COLUMNS if c != 'settlement_currency']
        miss = sorted(set(need) - have)
        if miss:
            raise RuntimeError(f'normalized_option_chain lacks {miss}')
        cols = [c for c in cols_wanted if c in have]
        grid = f'AND EXTRACT(minute FROM n.state_time) % {ls.ENTRY_GRID_MIN} = 0' if hourly else ''
        df = con.execute(f'''
            WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
            SELECT {', '.join('n.' + c for c in cols)}
            FROM normalized_option_chain n
            JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
            WHERE n.minutes_to_expiry BETWEEN 60 AND {ls.MAX_TTE_MIN} {grid} {where_extra}
        ''').df()
    finally:
        con.close()
    for c in cols_wanted:
        if c not in df:
            df[c] = None if c == 'settlement_currency' else np.nan
    return df

def settlement(st, cache_out, offline):
    sessions = sorted(st.session.unique())
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        idx = con.execute('SELECT state_time, underlying, futures_index_price FROM time_series_feature_state').df()
    finally:
        con.close()
    approx = ls.approx_settlement(idx, sessions)
    names = sorted({ls.index_name(u, c) for u, c in zip(st.underlying, st.settlement_currency)})
    seed = VRP / '02' / '02_delivery_prices.csv'
    if not cache_out.exists() and seed.exists():
        cache_out.write_text(seed.read_text())
    if offline:
        delivery = pd.read_csv(cache_out, dtype={'session': str}) if cache_out.exists() else pd.DataFrame(columns=['index_name', 'session', 'delivery_price'])
    else:
        delivery = ls.fetch_delivery_prices(names, cache_csv=cache_out)
    delivery = delivery[delivery.session.isin(sessions)]
    x = ls.attach_settlement(st, delivery, approx)
    print('settlement source:', x.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict())
    return x


## 1. Entries, settlement, forecast wedge, wings

In [3]:
chain = load_chain(list(ls.CHAIN_COLUMNS), 'AND abs(ln(n.strike / n.option_forward_price)) < 0.15', hourly=True)
legs = cs.prepare_legs(chain, confirm=CONFIRM)
del chain
st = ls.select_straddles(legs)
st = settlement(st, OUT / '04_delivery_prices.csv', OFFLINE)
st = st[np.isfinite(st.S_T)].reset_index(drop=True)
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ts = con.execute('SELECT state_time, underlying, return_5m FROM time_series_feature_state').df()
finally:
    con.close()
rets = cs.index_panel(ts)
if CONFIRM:
    spec = cs.load_frozen(SPEC)
    profiles, thr = spec['profiles'], spec['thresholds']
else:
    profiles = {a: cs.clock_profile(r) for a, r in rets.items()}
x = cs.attach_wedge(st, rets, profiles)
x = cs.attach_wings(x, legs)
if not CONFIRM:
    thr = cs.tercile_thresholds(x['wedge'])
x['tercile'] = cs.tercile(x['wedge'].to_numpy(float), thr)
print(f'entries {len(x):,} | sessions {x.session.nunique()} | wedge finite {np.isfinite(x.wedge).mean():.1%} | '
      f'iron fly available {x.K_up.notna().mean():.1%} | thresholds {np.round(thr, 3)}')
display(x.groupby('underlying').wedge.describe().round(3))

settlement source: {'exchange': 140}
entries 2,515 | sessions 20 | wedge finite 85.4% | iron fly available 91.9% | thresholds [-0.192  0.204]


,count,mean,std,min,25%,50%,75%,max
underlying,,,,,,,,
AVAX,319.0,-0.348,0.446,-1.400,-0.638,-0.333,-0.057,0.659
BTC,321.0,-0.291,0.466,-1.519,-0.564,-0.249,-0.016,1.011
ETH,321.0,-0.211,0.417,-1.475,-0.443,-0.168,0.042,0.780
HYPE,310.0,0.019,0.407,-1.790,-0.156,0.055,0.254,1.069
SOL,315.0,0.087,0.365,-0.832,-0.143,0.128,0.331,1.131
TRX,273.0,0.919,0.385,0.054,0.620,0.919,1.213,1.928
XRP,288.0,0.109,0.342,-0.794,-0.087,0.130,0.317,1.004


## 2. Outcomes by tercile

In [4]:
ex = x.join(cs.short_pnl(x, 'exec'))
mid = x.join(cs.short_pnl(x, 'mid'))
cols = ['straddle_ret_on_credit', 'straddle_pnl_bps_underlying', 'ironfly_ror', 'ironfly_pnl_bps_underlying']
t_exec = cs.tercile_table(ex, cols, n_boot=1000).assign(prices='executable')
t_mid = cs.tercile_table(mid, cols, n_boot=1000).assign(prices='mid')
tab = pd.concat([t_exec, t_mid], ignore_index=True)
display(tab.pivot_table(index=['prices', 'metric'], columns='tercile', values='M').round(4))
mono = {f'{p}_{c}': cs.top_minus_bottom(d, c, n_boot=N_BOOT) for p, d in (('exec', ex), ('mid', mid))
        for c in ('ironfly_ror', 'straddle_ret_on_credit')}
print('High minus low tercile (session-paired):'); print(json.dumps(mono, indent=2, default=float))
print('Iron fly credit as share of straddle credit (median):', round(float(ex.ironfly_credit_share.median()), 3))

tercile                                    high      low      mid
prices     metric                                                
executable ironfly_pnl_bps_underlying  -13.5532 -34.2518 -19.4333
           ironfly_ror                  -0.1133  -0.2096  -0.1689
           straddle_pnl_bps_underlying   7.0315 -25.9338   4.7999
           straddle_ret_on_credit        0.0497  -0.2858  -0.0896
mid        ironfly_pnl_bps_underlying    4.0526 -15.1211   2.7182
           ironfly_ror                   0.0364  -0.0972  -0.0422
           straddle_pnl_bps_underlying  18.2712 -13.9557  18.7879
           straddle_ret_on_credit        0.1905  -0.1110   0.0859

High minus low tercile (session-paired):
{
  "exec_ironfly_ror": {
    "n_sessions": 17,
    "M": 0.09630255090333047,
    "ci_low": -0.0241919687390037,
    "ci_high": 0.2216791249548846,
    "loso_min": 0.05517489581511478,
    "loso_max": 0.11647244646015706,
    "share_sessions_positive": 0.5882352941176471
  },
  "exec_straddle_ret_on_credit": {
    "n_sessions": 17,
    "M": 0.33553066156560035,
    "ci_low": 0.06559275882254831,
    "ci_high": 0.6132361624402545,
    "loso_min": 0.24730529792478176,
    "loso_max": 0.3927144822438395,
    "share_sessions_positive": 0.6470588235294118
  },
  "mid_ironfly_ror": {
    "n_sessions": 17,
    "M": 0.13359538800782877,
    "ci_low": 0.012636631599809956,
    "ci_high": 0.26496583448341093,
    "loso_min": 0.09088964412709541,
    "loso_max": 0.15281796485967103,
    "share_sessions_positive": 0.5882352941176471
  },
  "mid_straddle_ret_on_credit": {
    "n_sessions": 17,
    "M": 0.3014419613543185,
    "ci_low": 0.07183136433673933,
 

## 3. Decision statistic (binding only in confirmation mode)

In [5]:
dec = cs.decide(ex, n_boot=N_BOOT)
dec['mode'] = MODE
dec['binding'] = bool(CONFIRM)
print(json.dumps(dec, indent=2, default=float))
if FREEZE:
    spec = cs.freeze(SPEC, profiles, thr, dev_last_session=str(x.session.max()))
    print('FROZEN:', SPEC, '| thresholds', spec['thresholds'], '| dev last session', spec['dev_last_session'])

{
  "verdict": "INSUFFICIENT_SESSIONS",
  "n_sessions": 17,
  "M": -0.1132847501109557,
  "ci_low": -0.22270182341120345,
  "ci_high": -0.010555250765370814,
  "loso_min": -0.13981718838753368,
  "loso_max": -0.08205878377361397,
  "share_sessions_positive": 0.35294117647058826,
  "mode": "development",
  "binding": false
}


## 4. Outputs

In [6]:
tag = {'development': 'dev', 'freeze': 'dev', 'confirm': 'conf'}[MODE]
tab.to_csv(OUT / f'04_{tag}_tercile_table.csv', index=False, float_format='%.6g')
(OUT / f'04_{tag}_decision.json').write_text(json.dumps({'decision': dec, 'high_minus_low': mono, 'thresholds': thr,
                                                         'registration': cs.registration()}, indent=2, default=float))
x.groupby(['underlying', 'tercile']).size().rename('n').to_csv(OUT / f'04_{tag}_counts.csv')
meta = {'module': '04_conditional_short', 'version': cs.VERSION, 'mode': MODE, 'n_entries': int(len(x)),
        'sessions': [x.session.min(), x.session.max()], 'n_sessions': int(x.session.nunique()),
        'ironfly_available_share': float(x.K_up.notna().mean()), 'thresholds': thr, 'runtime_s': round(time.time() - T0)}
(OUT / f'04_{tag}_run_metadata.json').write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str))

{
  "module": "04_conditional_short",
  "version": "0.1.0",
  "mode": "development",
  "n_entries": 2515,
  "sessions": [
    "2026-09-21",
    "2026-10-10"
  ],
  "n_sessions": 20,
  "ironfly_available_share": 0.9188866799204771,
  "thresholds": [
    -0.19209099117703587,
    0.2044670332096173
  ],
  "runtime_s": 11
}
